# Check the carried-over claims: OCR, small crops, and diagrams

Notebook 01 states three common observations about parsing visual documents as hypotheses. This notebook measures all three: the first two on the FinanceBench filings, the third on the ViDoRe v3 physics slides. It reads `outputs/replication.json` and `outputs/diagrams.json`, which `replicate_claims.py` and `diagram_claim.py` write, and calls no model.

You will:

- Compare Docling's text layer with OCR on 300 born-digital filing pages: text, tables, retrieval, and answers.
- Check whether picture descriptions state more unsupported content on small crops, for three prompts.
- Measure how many of a slide's drawn relations the text layer, OCR, and vision-model descriptions state, and what descriptions do for page retrieval.

Each part is one run with no repeats, with the fixed sampling seed 0.

In [1]:
import json
import sys
from pathlib import Path


sys.path.insert(0, str(Path.cwd()))

# Off reads the committed results. On rebuilds them: replicate_claims.py takes
# hours of CPU parsing, on-prem reading, and about $0.85 of judge calls;
# diagram_claim.py needs a local snapshot of vidore/vidore_v3_physics (it
# downloads one) and about $7 of participant-model calls.
RUN_EXPENSIVE = False

if RUN_EXPENSIVE:
    import diagram_claim
    import replicate_claims

    replicate_claims.main(["parse", "fidelity", "retrieval", "answers", "crops", "report"])
    diagram_claim.main(["images", "ocr", "select", "describe", "answer", "judge", "relations", "retrieval", "report"])

result = json.loads(Path("outputs/replication.json").read_text(encoding="utf-8"))
diagram_result = json.loads(Path("outputs/diagrams.json").read_text(encoding="utf-8"))
print(json.dumps(result["produced_by"], indent=1))

{
 "model": "Qwen3_8-27B",
 "producer": "implementations/special_topics/visual_documents/replicate_claims.py",
 "mode": "explore: one run, fixed sampling seed 0",
 "parser": "docling (table structure on); RapidOCR PP-OCRv6 small, English, for OCR conditions",
 "raster_dpi": 216,
 "embeddings": "BAAI/bge-small-en-v1.5 (local)",
 "reader": "Qwen3_8-27B",
 "answer_judge": "Qwen3_6-35B-A3B",
 "describer": "Qwen3_8-27B",
 "description_judge": "gemini-3.8-flash",
 "endpoint": "vector-cluster (reader, describer, answer judge); vector-proxy-paid (description judge)"
}


## Text layer against OCR on born-digital pages

The claim: on born-digital PDFs the text layer beats OCR, and forcing OCR only lowers quality.

Every condition parses the same page subsets with Docling, table structure on, and is compared with a re-parse with OCR off, the setting RI #3's ingestion uses:

- `ocr-forced` renders each page to an image at 216 dpi and parses it with full-page RapidOCR, so no character comes from the text layer. It runs on 300 pages: the gold page of every question in notebook 01's set, plus seeded distractors. Parsing all 1,000 pages with OCR did not fit the run's machine time.
- `ocr-auto` is Docling's own default, OCR only on regions the text layer does not cover, on the 103 figure pages, where bitmaps sit.
- `flag-forced` is Docling's full-page OCR mode on the original PDF, on 40 sampled pages.

Word accuracy is the share of the text layer's words that align to the same word in the condition's text, and kept numbers the share of its numeric tokens the condition reproduces.

In [2]:
fidelity = result["claim_1_text_layer_vs_ocr"]["fidelity"]
print(
    f"{'condition':<13}{'pages':>6}{'identical':>10}{'words':>8}{'numbers':>9}{'extra/page':>11}{'tables':>9}{'cells kept':>11}"
)
for name, row in fidelity.items():
    print(
        f"{name:<13}{row['pages']:>6}{row['identical_pages']:>10}{row['mean_word_accuracy']:>8.3f}"
        f"{row['numbers_kept']:>9.3f}{row['numbers_extra_per_page']:>11.2f}"
        f"{str(row['tables']) + '/' + str(row['tables_ref']):>9}{row['table_cells_kept']:>11.3f}"
    )
print()
for name, row in fidelity.items():
    print(
        f"{name}: {row['pages_without_text_layer']} pages without a text layer, {row['chars_added_on_pages_without_text_layer']} characters added there"
    )

condition     pages identical   words  numbers extra/page   tables cells kept
ocr-auto        103        82   0.994    0.997       1.26    45/45      1.000
ocr-forced      300        33   0.953    0.981       0.87  245/249      0.978
flag-forced      40        12   0.967    0.975       0.08    16/16      1.000

ocr-auto: 3 pages without a text layer, 120 characters added there
ocr-forced: 3 pages without a text layer, 318 characters added there
flag-forced: 1 pages without a text layer, 14 characters added there


Forced OCR reproduces 95% of the text layer's words and 98% of its numbers, adds about one stray number per page, finds 245 of 249 tables, and keeps 97.8% of their numeric cells. On a born-digital filing that is a small loss and no gain: only 3 of the 300 pages lack a text layer, and OCR adds 318 characters across them. Docling's default (`ocr-auto`) leaves 82 of 103 figure pages identical to the text layer and changes the rest by a few words (word accuracy 0.994).

In [3]:
claim = result["claim_1_text_layer_vs_ocr"]
groups = ["figure-only", "duplicated", "table", "control", "null", "all"]
print(f"{'parse':<16}" + "".join(f"{g:>13}" for g in groups))
for name in ("text-layer", "ocr-forced"):
    recall = claim["page_recall_at_5"][name]
    cells = [recall[g]["recall_at_5"] for g in groups]
    print(f"{name + ' r@5':<16}" + "".join(f"{c:>13.3f}" if c is not None else f"{'-':>13}" for c in cells))
for name in ("text-layer", "ocr-forced"):
    answers = claim["answers"][name]
    print(
        f"{name + ' correct':<16}"
        + "".join(f"{str(answers[g]['correct']) + '/' + str(answers[g]['n']):>13}" for g in groups)
    )
print(claim["paired"])

parse             figure-only   duplicated        table      control         null          all
text-layer r@5          0.711        0.932        0.783        0.650            -        0.764
ocr-forced r@5          0.697        0.932        0.767        0.625            -        0.750
text-layer correct         5/76        37/44        36/60         4/40        26/26      108/246
ocr-forced correct         6/76        33/44        32/60         5/40        26/26      102/246
{'text-layer_vs_ocr-forced': {'text_layer_only': 10, 'condition_only': 4, 'p': 0.1796}}


Both parses feed the same pipeline: arm `T` of notebook 01, `bge-small-en-v1.5` over table-aware chunks of the 300 pages, the top five chunks to `Qwen3_8-27B`, and the reference judge. Page recall at 5 falls from 0.764 to 0.750 and correct answers from 108 to 102 of 246; the text layer alone answers 10 questions forced OCR misses, and OCR alone 4 (exact McNemar p = 0.18). Most of the loss is on table and duplicated-chart questions, where a misread digit changes the answer.

The claim replicates in direction and is small here: on clean renders of born-digital filings, OCR costs about 2 points of retrieval and 2 points of accuracy, and buys nothing. The run cannot separate the answer difference from noise. For OCR to win, pages would need text the layer lacks: scans, image-only infographics, or text drawn as vector paths. The cheapest test is the same comparison on the 3 image-only pages plus scanned copies of 20 pages at 150 dpi.

## Unsupported content in picture descriptions by crop size

The claim: a generic picture prompt on a small crop describes things the picture does not contain, so domain prompts and a minimum crop size matter.

Pictures from the 49 filings in notebook 01's universe are binned by their share of the page, 30 per bin with seed 0, and described by `Qwen3_8-27B` with three prompts: notebook 01's generic prompt (`E1`), its domain prompt (`E2`), and a third, the generic prompt plus the domain prompt's grounding lines ("Use only what the image shows") and a one-line rule for logos. `gemini-3.8-flash` then splits each description into claims and marks each supported or unsupported by the crop. "Any" is the share of descriptions with at least one unsupported claim; "share" is unsupported claims over checked claims.

In [4]:
claim = result["claim_2_small_crops"]
prompts = ["generic", "domain", "grounded-generic"]
print(f"{'bin':<8}" + "".join(f"{p + ' any':>22}{'share':>8}" for p in prompts))
for name, row in {**claim["per_bin"], "all": claim["all"]}.items():
    print(f"{name:<8}" + "".join(f"{row[p]['any_unsupported']:>22.2f}{row[p]['claim_share']:>8.3f}" for p in prompts))
print(
    "claims per description, all bins: "
    + ", ".join(f"{p} {claim['all'][p]['claims_per_description']}" for p in prompts)
)
print("hand check:", claim.get("hand_check"))

bin                generic any   share            domain any   share  grounded-generic any   share
<0.5%                     0.10   0.014                  0.03   0.007                  0.43   0.103
0.5-1%                    0.37   0.090                  0.23   0.056                  0.33   0.055
1-5%                      0.27   0.063                  0.10   0.016                  0.17   0.025
5-10%                     0.37   0.059                  0.10   0.009                  0.33   0.043
>=10%                     0.47   0.091                  0.30   0.079                  0.40   0.143
all                       0.31   0.064                  0.15   0.041                  0.33   0.080
claims per description, all bins: generic 7.15, domain 8.37, grounded-generic 7.53
hand check: {'pictures': 12, 'descriptions': 36, 'judge_flags_checked': 9, 'flags_confirmed': 8, 'flags_rejected': 1, 'missed_unsupported_found': 0, 'note': "12 crops viewed by hand against all three descriptions: 5 from the

The small-crop half of the claim does not replicate on this vision model. With the generic prompt, unsupported content is rarest on the smallest crops and most common on the largest: 0.10 of descriptions under 0.5% of the page carry an unsupported claim, against 0.47 at 10% or more. Small crops in filings are logos, page furniture, and portraits, with few checkable claims; large crops are charts, where a wrong legend style or a miscounted series is easy to state. The smallest bin is also narrow: 26 of its 30 crops are the same payment-company logo, repeated across one filing's pages.

The domain half does replicate: the domain prompt halves the share of descriptions with an unsupported claim, 0.15 against 0.31 for the generic prompt, while stating more claims (8.4 against 7.2 per description). The grounding lines alone do not do it: the grounded generic prompt scores 0.33, and 0.43 on the smallest bin, where it describes logo details it gets wrong. What helps is the domain prompt's structure (type first, then labels and values), not the instruction to stay inside the image.

A hand check of 12 crops against all 36 descriptions confirmed 8 of the 9 claims the judge marked unsupported (wrong line styles, a miscounted set of circles, wrong logo details) and found no unsupported claim it missed in the text read.

This fits the earlier result that the generic prompt beat the domain prompt on answering, 19 to 5: that loss was retrieval, because `E2` writes markdown value tables that `bge-small-en-v1.5` matches to questions less well than prose. The domain prompt is the more faithful description and the worse retrieval text. For small crops to matter, the vision model would have to fill in content it cannot see at low resolution; with `CROP_SCALE` 2.5 even 40-pixel logos render legibly. The cheapest test is the same bins at crop scale 1.0 and a second vision model, about 300 local description calls and 300 judge calls.

## Datasets for the claim about diagrams

The third claim, that OCR recovers almost nothing from diagrams while a vision-model description recovers sequence and connections, needs diagram-heavy pages, which the filings lack. The ViDoRe v3 benchmark on Hugging Face has eight subsets; the English query counts below are one sixth of each subset's queries, which repeat in six languages. "Visual" counts English queries labelled infographic or image.

| Subset | Pages | Documents | English queries | Visual | Licence of the documents |
| --- | ---: | ---: | ---: | ---: | --- |
| `vidore_v3_hr` | 1,110 | 14 reports | 318 | 41 | CC BY 4.0 |
| `vidore_v3_industrial` | 5,244 | 27 technical manuals | 283 | 75 | US Air Force technical orders, public release |
| `vidore_v3_physics` | 1,674 | 42 slide decks | 302 | 184 | CC BY 4.0 |
| `vidore_v3_pharmaceuticals` | 2,313 | 52 decks and books | 364 | 144 | US FDA, public domain |
| `vidore_v3_computer_science` | 1,360 | 2 textbooks | 215 | 52 | OpenStax, CC BY 4.0 |

`vidore_v3_hr`'s visuals are mostly charts, so it tests only the chart half of the claim. The physics slides, French course decks full of flow diagrams, apparatus sketches, and concept maps, are the closest to diagram-heavy, and the next section uses them.

## Diagrams on the ViDoRe v3 physics slides

The claim: OCR returned almost nothing useful on diagrams, while a vision-model description recovered sequence and connection information.

Three designs were considered before running. Answering the dataset's visual queries from each representation of the gold page tests the claim end to end. Scoring each representation on the relations a page draws tests the mechanism directly. Ranking all 1,674 pages with and without descriptions tests whether it matters for retrieval. All three ran; the first turned out to be the wrong instrument, as the last cell of this section explains.

Each page is turned into text seven ways: the PDF text layer (pypdfium2); RapidOCR lines; the same lines with their boxes, so a reader can use the layout; `gemini-3.5-flash-lite` with notebook 01's generic prompt; `gemini-3.5-flash-lite` and `claude-haiku-5-5` with a relations prompt that asks for every arrow, line, and order as "A -> B: meaning"; and the relations prompt on a copy of the page where everything outside the OCR boxes is painted white. That last one is a control: it keeps the labels and their positions and removes the drawing.

The test pages are the gold pages of the English queries that `Qwen3_8-27B` (on-prem) marks as needing a relation the drawing shows, plus 20 seeded pages from the other picture queries. `claude-sonnet-5-5` lists the relations each page draws, and `gemini-3.8-flash` marks which of them each representation states.

In [5]:
selection = diagram_result["selection"]
print(
    f"{selection['diagram_relation']} of {selection['picture_pairs']} picture queries need a drawn relation: {selection['kinds']}"
)
print("hand check of the relation lists:", diagram_result.get("hand_check"))
print()
names = list(diagram_result["relations"]["diagram"]["recall"])
print(f"{'representation':<22}" + "".join(f"{g + ' recall':>24}{'all':>6}" for g in diagram_result["relations"]))
for name in names:
    row = "".join(
        f"{block['recall'][name]['recall']:>24.3f}{block['recall'][name]['all_stated']:>6.2f}"
        for block in diagram_result["relations"].values()
    )
    print(f"{name:<22}" + row)
for group, block in diagram_result["relations"].items():
    print(f"{group}: {len(block['pages'])} pages, {block['relations_per_page']} drawn relations per page")

34 of 137 picture queries need a drawn relation: {'sequence': 9, 'connection': 14, 'arrangement': 11}
hand check of the relation lists: {'pages': 2, 'relations_listed': 20, 'relations_confirmed': 19, 'relations_arguable': 1, 'note': "Two diagram pages (an energy-and-entropy flow figure and a roadmap of linked boxes) viewed against the lister's 20 relations; 19 match the drawing and one gives an arrow's start loosely. On the text-only copy of the roadmap page, the short arrows between stacked boxes survive inside the merged OCR boxes, so the text-only control keeps some drawing."}

representation                  diagram recall   all    other-picture recall   all
text-layer                               0.019  0.00                   0.029  0.00
ocr                                      0.023  0.00                   0.069  0.05
ocr-boxes                                0.053  0.00                   0.069  0.00
vlm-generic                              0.185  0.00                   0.115  0.

On the 30 diagram pages (265 drawn relations, 8.8 per page), the text layer states 2% of the relations and OCR 2%; OCR with its boxes reaches 5%. A description with the relations prompt states 55% on `gemini-3.5-flash-lite` and 64% on `claude-haiku-5-5`. The claim holds on these slides in this form: OCR recovers almost none of the sequence and connection information, and a vision model that is asked for it recovers about half.

The surprise is the prompt. With notebook 01's generic prompt, the same vision model states 18% of the relations, a third of what the relations prompt gets: a generic description names what is on the slide and skips how the parts connect. The vision model alone does not carry the claim; the request for relations does.

The text-only control separates reading the drawing from inferring from the labels. With the drawing painted out, the relations prompt falls from 55% to 19%, so about two thirds of what it recovers comes from the arrows and lines. The control leaks a little: short arrows inside merged OCR boxes survive, so 19% is an upper bound on what the labels alone give.

The 20 control pages, whose queries the selector did not mark, draw as many relations (8.7 per page) and give the same ordering. The selector judged the query, not the page; on these slides almost every picture page is a diagram. Two caveats: `claude-sonnet-5-5` listed the relations, the same family as `claude-haiku-5-5`, which may flatter the haiku row, and a hand check of 2 pages confirmed 19 of the 20 listed relations.

In [6]:
retrieval = diagram_result["retrieval"]
print(f"{retrieval['pages']} pages; queries per group: {retrieval['groups']}")
groups = ["diagram", "other-picture", "text", "all"]
print(f"{'ranking':<34}" + "".join(f"{g + '@5':>17}" for g in groups))
for name, recall in retrieval["recall"].items():
    print(f"{name:<34}" + "".join(f"{recall[g + '@5']:>17.3f}" for g in groups))

1674 pages; queries per group: {'diagram': 34, 'other-picture': 103, 'text': 141, 'all': 302}
ranking                                   diagram@5  other-picture@5           text@5            all@5
dense|text-layer                              0.441            0.544            0.503            0.510
bm25|text-layer                               0.529            0.738            0.667            0.666
dense|text-layer+vlm-relations                0.618            0.631            0.539            0.586
bm25|text-layer+vlm-relations                 0.559            0.738            0.681            0.679


Every one of the 1,674 pages was described with the relations prompt on `gemini-3.5-flash-lite`, and each page's text layer and description pieces were indexed side by side. With `bge-small-en-v1.5` and the English queries, page recall at 5 on the 34 diagram queries rises from 0.44 to 0.62, against 0.50 to 0.54 on 141 text-only queries. Part of that is language: 89% of the descriptions came out in English, the queries are English, and the slides are French. With BM25 and the French versions of the queries, the gain shrinks to 0.53 to 0.56 on diagram queries and 0.67 to 0.68 on text queries.

The first design, answering each query from one gold page, does not test the claim on this dataset. ViDoRe v3 physics queries are open-ended and their reference answers draw on several pages: only 60 of the 302 English queries have a single gold page, and 5 of the 34 diagram queries. `gemini-3.5-flash-lite` answers "NOT STATED" to most of them from any one page, its own page image included, so `outputs/diagrams.json` keeps those counts (`groups`) but they rank nothing.

For OCR to recover relations, the arrows would have to be text: a flow drawn as "A → B" in a text box. The cheapest test of the opposite direction is whether a relations-prompt description helps a reader on questions about one diagram; that needs single-page diagram questions, written by hand for about 30 of these pages.